# Part 1 - LangChain Requirements Agents

## An Agentic AI Requirements-to-Delivery Assistant for Dynamics 365 F&O

Business requests for Dynamics 365 Finance and Operations often arrive as informal emails with missing
business rules, affected modules, exceptions, security requirements and expected outcomes. This causes
repeated clarification, inconsistent documentation and development rework.

This notebook builds the **first three agents** of the system with **LangChain** chains and structured outputs:

1. **Requirements Agent** - extracts actors, rules, scope, exceptions and expected outcomes.
2. **Gap and Retrieval Agent (classification)** - classifies the request as Finance, Supply Chain, functional, technical or mixed.
3. **Gap and Retrieval Agent (gaps)** - identifies ambiguities and prepares clarification questions.

Finally, the chains are connected into a small **LangGraph** pipeline.

| Notebook | Part |
|---|---|
| **01** | **LangChain requirements agents (this notebook)** |
| 02 | RAG knowledge base + knowledge MCP server |
| 03 | Email: authorized threads, clarification drafts, customer answers |
| 04 | Azure DevOps work items + owner recommendation |
| 05 | Functional / technical design documents and test cases |
| 06 | Validation, traceability and reflexion |
| Final | Supervisor workflow that incorporates all parts |

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook only needs the chat model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`); Ollama is then not needed for this notebook. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [1]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = LLM_PROVIDER == "ollama"  # this notebook only needs the chat model


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

Chat model: openai / gpt-6-luna


In [ ]:
# 2. Pull the chat model and the embedding model (only when the agents run on Ollama)
if NEEDS_OLLAMA:
    !ollama pull {LLM_MODEL}
    !ollama pull {EMBED_MODEL}
    !ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [2]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

LangSmith tracing OFF


In [3]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [4]:
from typing import TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

In [5]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))

## The Input Request

The initial version of the system works on **one manually supplied requirement at a time** (proposal scope).
The request below is typical: it has a real business need, but the amount, companies, approver details,
the meaning of "blocked" and the deadline are all vague.

In [6]:
REQUEST_ID = "CR-2026-001"

CUSTOMER_REQUEST = """
Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group
"""
print(CUSTOMER_REQUEST)


Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group



## Why a Single Prompt Is Not Enough

The proposal argues that a standalone LLM *"may silently assume missing facts"*. To show this, the model is
asked to go straight from the email to user stories in one prompt.

**Prediction:** the model will produce confident user stories with invented details (an amount, a list of
companies, an approver) that the customer never gave.

In [7]:
single_prompt_answer = ask_text(
    "You are a Dynamics 365 F&O consultant.",
    f"Write the user stories with acceptance criteria for this request:\n{CUSTOMER_REQUEST}",
)
print(single_prompt_answer)

## Epic: Strengthen vendor invoice approval and payment controls

### Story 1 - Require finance manager approval above a threshold

**As an** Accounts Payable user,  
**I want** vendor invoices above a configurable amount to require finance manager approval,  
**So that** higher-value invoices receive the required financial oversight before posting.

**Acceptance criteria**

1. The approval threshold can be configured for each in-scope legal entity.
2. When an invoice exceeds the configured threshold, submitting it starts the vendor invoice approval workflow and routes it to the designated finance manager for that legal entity.
3. The approver can approve or reject the invoice, and the decision and any comments are recorded in the workflow history.
4. An invoice that is rejected cannot be posted until it is corrected and resubmitted for approval.
5. An invoice at or below the threshold does not require this approval, unless another configured workflow requires it.
6. The threshold is e

## Structured Schemas

Instead of free text, every agent returns a **Pydantic** object. These objects become part of the
structured package that is handed from agent to agent: requirement IDs, source text, classification,
gaps and (later) confirmed answers, assumptions, evidence and approval status.

In [8]:
from typing import Literal

from pydantic import BaseModel, Field


class Requirement(BaseModel):
    req_id: str = Field(description="Stable ID such as REQ-001")
    statement: str = Field(description="One clear, testable requirement sentence")
    source_text: str = Field(description="The customer's words this requirement comes from")
    actor: str = Field(description="Who performs or benefits from the requirement")
    business_rules: list[str] = Field(description="Rules, thresholds or conditions that are actually stated")
    exceptions: list[str] = Field(description="Exceptions or edge cases that are actually stated")
    expected_outcome: str = Field(description="What should happen when the requirement is met")


class RequirementExtraction(BaseModel):
    summary: str = Field(description="Two-sentence summary of the request")
    actors: list[str]
    in_scope: list[str]
    out_of_scope: list[str]
    security_constraints: list[str] = Field(description="Security, approval or segregation-of-duties constraints")
    requirements: list[Requirement]


class Classification(BaseModel):
    module: Literal["Finance", "Supply Chain", "Finance and Supply Chain", "Out of scope"]
    sub_areas: list[str] = Field(description="F&O areas such as Accounts payable, Procurement, Inventory")
    work_type: Literal["Functional", "Technical", "Mixed"]
    design_documents: Literal["FDD", "TDD", "FDD and TDD"]
    risk_level: Literal["Low", "Medium", "High"]
    rationale: str


class Gap(BaseModel):
    gap_id: str = Field(description="Stable ID such as GAP-01")
    req_id: str = Field(description="The requirement this gap belongs to")
    missing_information: str
    why_it_matters: str
    clarification_question: str = Field(description="One clear question the customer can answer")
    assumption_if_unanswered: str


class GapAnalysis(BaseModel):
    gaps: list[Gap]

## Agent Prompts

Each agent has its own role prompt. The key rule for the Requirements Agent is that it must **never invent
values**; anything missing becomes a gap for the next agent.

In [9]:
REQUIREMENTS_SYSTEM = """
You are the Requirements Agent of a Dynamics 365 Finance and Operations (F&O) delivery team.
Turn an informal customer request into structured requirements.

Rules:
- Write one requirement per distinct need and number them REQ-001, REQ-002, ...
- source_text must quote the customer's own words.
- Record only rules, thresholds, exceptions and outcomes that are actually stated.
  Never invent values such as amounts, companies or roles. Missing details are handled later as gaps.
- Put approval, security and segregation-of-duties constraints in security_constraints.
"""

REFINE_SYSTEM = """
You are the Requirements Agent of a Dynamics 365 F&O delivery team.
Update the requirements with the customer's confirmed answers.

Rules:
- Keep the existing REQ IDs; add a new ID only for a genuinely new need the customer confirmed.
- Put confirmed values (amounts, currencies, legal entities, roles, days, dates) into the statements
  and business rules, using the customer's exact values.
- Do not present assumptions as confirmed facts.
"""

CLASSIFY_SYSTEM = """
You are the Gap and Retrieval Agent of a Dynamics 365 F&O delivery team. Classify the request.

- Finance: general ledger, accounts payable, accounts receivable, cash and bank, fixed assets, budgeting.
- Supply Chain: procurement, inventory, warehouse, sales orders, production.
- Functional work = configuration and process only. Technical work = code, integrations, batch jobs, reports.
  Mixed = both.
- design_documents: FDD for functional work, TDD for technical work, "FDD and TDD" for mixed work.
Use the retrieved knowledge as evidence.
"""

GAPS_SYSTEM = """
You are the Gap and Retrieval Agent of a Dynamics 365 F&O delivery team.
Find the missing information that would force a consultant or developer to guess.

Check every requirement for: amounts, thresholds and currency; legal entities in scope; approver roles
and escalation; exceptions; existing versus new data; security and segregation of duties; audit needs;
deadlines.
- Number the gaps GAP-01, GAP-02, ... (at most 8).
- Write each clarification question in plain business language, without X++ or technical jargon.
- Give a safe assumption to use if the customer does not answer.
"""

## Requirements Agent (LangChain chain)

The chain is `prompt | llm.with_structured_output(RequirementExtraction)`.

`with_structured_output(..., method="json_schema")` sends the Pydantic JSON schema to Ollama, so the model is
constrained to return valid JSON that LangChain parses directly into the Pydantic object.

In [10]:
requirements_prompt = ChatPromptTemplate.from_messages([
    ("system", REQUIREMENTS_SYSTEM),
    ("human", "Customer request:\n{request}"),
])

requirements_chain = requirements_prompt | llm.with_structured_output(RequirementExtraction, method="json_schema").with_config(run_name="Requirements Agent")

extraction = requirements_chain.invoke({"request": CUSTOMER_REQUEST})
show("REQUIREMENTS AGENT", extraction)


========== REQUIREMENTS AGENT ==========

{
  "summary": "The customer needs vendor invoices above an unspecified amount approved by the finance manager before posting, and vendors without bank details blocked to prevent mistaken payments. The changes should cover all companies and be live before quarter close.",
  "actors": [
    "Finance manager",
    "Accounts payable users",
    "Vendor payment process"
  ],
  "in_scope": [
    "Vendor invoice approval before posting",
    "Blocking vendors without bank details, including existing vendors",
    "All companies",
    "Go-live before quarter close"
  ],
  "out_of_scope": [],
  "security_constraints": [
    "Vendor invoices above the applicable amount must receive finance manager approval before they can be posted.",
    "The current ability of anyone in AP to post invoices must not allow invoices requiring approval to be posted without that approval."
  ],
  "requirements": [
    {
      "req_id": "REQ-001",
      "statement": "Requi

## Gap and Retrieval Agent - Classification

The classification decides which design documents are needed later (FDD, TDD or both) and which part of
the knowledge base should be searched. Knowledge retrieval itself is added in notebook 02.

In [11]:
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", CLASSIFY_SYSTEM),
    ("human", "Requirements:\n{requirements}\n\nRetrieved knowledge:\n{evidence}"),
])

classify_chain = classify_prompt | llm.with_structured_output(Classification, method="json_schema").with_config(run_name="Classification")

classification = classify_chain.invoke({
    "requirements": extraction.model_dump_json(indent=2),
    "evidence": "No knowledge retrieved in this notebook (RAG is added in notebook 02).",
})
show("CLASSIFICATION", classification)


========== CLASSIFICATION ==========

{
  "module": "Finance",
  "sub_areas": [
    "Accounts payable",
    "Vendor management",
    "Vendor payments"
  ],
  "work_type": "Functional",
  "design_documents": "FDD",
  "risk_level": "High",
  "rationale": "The request concerns accounts payable invoice approval and vendor payment controls, which are Finance processes. It calls for approval workflow and vendor-blocking configuration across all companies, with no code, integration, batch job, or report requested. Risk is high because the approval threshold is unspecified, the controls must prevent unauthorized posting and mistaken payment, existing vendors need to be addressed, and delivery is required before quarter close. No retrieved knowledge was available; this classification is based on the supplied requirements."
}


## Gap and Retrieval Agent - Gaps and Clarification Questions

For every requirement, the agent checks what a consultant or developer would otherwise have to guess, and
turns each gap into a customer-friendly question with a safe fallback assumption.

In [12]:
gaps_prompt = ChatPromptTemplate.from_messages([
    ("system", GAPS_SYSTEM),
    ("human", "Original request:\n{request}\n\nRequirements:\n{requirements}\n\nClassification:\n{classification}"),
])

gaps_chain = gaps_prompt | llm.with_structured_output(GapAnalysis, method="json_schema").with_config(run_name="Gap Agent")

gap_analysis = gaps_chain.invoke({
    "request": CUSTOMER_REQUEST,
    "requirements": extraction.model_dump_json(indent=2),
    "classification": classification.model_dump_json(indent=2),
})

print("========== CLARIFICATION QUESTIONS ==========\n")
for gap in gap_analysis.gaps:
    print(f"{gap.gap_id} ({gap.req_id}) {gap.clarification_question}")
    print(f"    if unanswered, assume: {gap.assumption_if_unanswered}\n")

========== CLARIFICATION QUESTIONS ==========

GAP-01 (REQ-001) What is the approval threshold and currency, and should it apply to the invoice total including tax, using the invoice currency or the company’s accounting currency?
    if unanswered, assume: Do not enable posting for invoices until the customer confirms the threshold and amount basis.

GAP-02 (REQ-001) Which finance manager roles or people may approve invoices, and who should approve when they are unavailable?
    if unanswered, assume: Only customer-designated finance managers may approve; do not allow AP users to approve on their behalf.

GAP-03 (REQ-001) Must the approver be different from the person who prepared the invoice, and are any approval bypasses or emergency overrides permitted?
    if unanswered, assume: Require an independent finance manager approval before posting; allow no bypass or override.

GAP-04 (REQ-001) What approval history or evidence must be retained for audit, and for how long?
    if unanswer

## Connecting the Chains with LangGraph

The three chains are now wrapped as nodes of a LangGraph pipeline:

**request -> extract requirements -> classify -> find gaps**

Each node reads the shared state and adds its own structured output, so later nodes always receive the
earlier results. To show that the pipeline generalises, it is run on a different, **Supply Chain** request.

In [13]:
class AnalysisState(TypedDict, total=False):
    request: str
    extraction: dict
    classification: dict
    gaps: list


def extract_node(state: AnalysisState) -> dict:
    result = requirements_chain.invoke({"request": state["request"]})
    print(f"[extract] {len(result.requirements)} requirements")
    return {"extraction": result.model_dump()}


def classify_node(state: AnalysisState) -> dict:
    result = classify_chain.invoke({
        "requirements": json.dumps(state["extraction"], indent=2),
        "evidence": "No knowledge retrieved in this notebook.",
    })
    print(f"[classify] {result.module} / {result.work_type} -> {result.design_documents}")
    return {"classification": result.model_dump()}


def gaps_node(state: AnalysisState) -> dict:
    result = gaps_chain.invoke({
        "request": state["request"],
        "requirements": json.dumps(state["extraction"], indent=2),
        "classification": json.dumps(state["classification"], indent=2),
    })
    print(f"[gaps] {len(result.gaps)} gaps")
    return {"gaps": [g.model_dump() for g in result.gaps]}


graph = StateGraph(AnalysisState)
graph.add_node("extract_requirements", extract_node)
graph.add_node("classify", classify_node)
graph.add_node("find_gaps", gaps_node)
graph.add_edge(START, "extract_requirements")
graph.add_edge("extract_requirements", "classify")
graph.add_edge("classify", "find_gaps")
graph.add_edge("find_gaps", END)

analysis_pipeline = graph.compile()
print(analysis_pipeline.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract_requirements(extract_requirements)
	classify(classify)
	find_gaps(find_gaps)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract_requirements;
	classify --> find_gaps;
	extract_requirements --> classify;
	find_gaps --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [14]:
SECOND_REQUEST = """
Hello,

Our warehouse keeps shipping batches that are already expired. The system should stop us from
picking expired batches and the quality team wants to be warned before a batch expires.
This is for the main distribution centre only.

Thanks,
Omar - Warehouse Manager
"""
analysis = analysis_pipeline.invoke({"request": SECOND_REQUEST})

show("SUPPLY CHAIN REQUEST - REQUIREMENTS", [r["statement"] for r in analysis["extraction"]["requirements"]])
show("SUPPLY CHAIN REQUEST - CLASSIFICATION", analysis["classification"])
show("SUPPLY CHAIN REQUEST - QUESTIONS", [g["clarification_question"] for g in analysis["gaps"]])

[extract] 2 requirements
[classify] Supply Chain / Mixed -> FDD and TDD
[gaps] 8 gaps

========== SUPPLY CHAIN REQUEST - REQUIREMENTS ==========

[
  "The system must prevent warehouse staff from picking expired batches at the main distribution centre.",
  "The system must warn the quality team before a batch expires at the main distribution centre."
]

========== SUPPLY CHAIN REQUEST - CLASSIFICATION ==========

{
  "module": "Supply Chain",
  "sub_areas": [
    "Inventory management",
    "Warehouse management",
    "Batch tracking and expiration",
    "Notifications"
  ],
  "work_type": "Mixed",
  "design_documents": "FDD and TDD",
  "risk_level": "Medium",
  "rationale": "Preventing expired-batch picking at the main distribution centre involves functional inventory and warehouse rules. Proactively warning the quality team before expiry may require a scheduled notification or other technical implementation, so the request is mixed. Scope is limited to the main distribution centre. N

## Observation

The single-prompt answer produced ready-looking user stories, but it had to fill the gaps itself: values
such as the approval amount, the companies in scope or the exact meaning of "blocked" were either invented or
left vague without being flagged. That is exactly the failure described in the proposal.

The chained agents behave differently. The Requirements Agent keeps each requirement traceable to the
customer's words (`source_text`) and records only what was stated. The classification selects the module,
work type and the design documents that will be needed. The Gap Agent turns every missing detail into a
numbered question with a fallback assumption instead of guessing.

Running the same pipeline on the Supply Chain request shows that the structure is not tied to one example.

These structured outputs are the start of the delivery package. The next notebooks add retrieved knowledge
(02), the customer email loop (03), Azure DevOps work items (04), design documents and tests (05) and
validation (06).